# App-16-Crossword-CSP (C#)

**Navigation** : [<< App-15-SportsScheduling](App-15-SportsScheduling-Csharp.ipynb) | [Index](../README.md) | [App-17-VRP >>](../Hybrid/App-17-VRP-Logistics.ipynb)

**Twin C# (.NET Interactive)** de [App-16-Crossword-CSP.ipynb](App-16-Crossword-CSP.ipynb) — marathon **#4956** (parite .NET <-> Python).

La generation de mots croises est un **CSP** (Constraint Satisfaction Problem) classique : on dispose d'une grille (cases blanches/noires), d'un dictionnaire de mots indexes par longueur, et on doit placer un mot dans chaque *slot* (suite maximale de cases blanches consecutives) de sorte que les **intersections** entre slots horizontaux et verticaux soient compatibles (meme lettre a la case de croisement).

## Plan pedagogique

1. **Modelisation** — CrosswordGrid, extraction des slots
2. **Backtracking** — assignation ordonnee (MRV), verification des intersections
3. **Propagation de contraintes** — forward-checking
4. **Generation aleatoire** + analyse
5. **Exercices**

> **Parite #4956** : la version Python s'appuie sur OR-Tools CP-SAT (solveur boite noire) et implemente aussi backtracking + propagation from-scratch. Ce twin C# (BCL .NET 9, **0 NuGet**) deroule le backtracking + forward-checking from-scratch (les internes pedagogiques que masque OR-Tools). La viz matplotlib devient un rendu ASCII de la grille (convention GT-4c).


## 1. Modelisation du probleme

Une **grille** de mots croises est une matrice de cases, chacune **blanche** (lettre a placer) ou **noire** (bloque). Un **slot** est une suite maximale de cases blanches consecutives (longueur >= 2), horizontalement ou verticalement. Chaque slot recoit un mot du dictionnaire de la bonne longueur. Deux slots qui se croisent a une case partagent une lettre : la position de cette lettre dans le mot horizontal doit egaler celle dans le mot vertical.

In [1]:
using System.Linq;
using System.Text;
using System.Collections.Generic;

static void Show(string s) { s.Display(); }

// Grille : true = case blanche (lettre), false = case noire (bloque).
public class CrosswordGrid
{
    public int Rows, Cols;
    public bool[,] White;   // White[r,c] = true si case a remplir
    public CrosswordGrid(bool[,] white)
    {
        White = white; Rows = white.GetLength(0); Cols = white.GetLength(1);
    }
    public bool IsWhite(int r, int c) => r>=0 && r<Rows && c>=0 && c<Cols && White[r,c];
}

// Slot : suite maximale de cases blanches consecutives.
// Direction : 'H' (horizontal, ligne fixe) ou 'V' (vertical, colonne fixe).
public record Slot(int Row, int Col, char Dir, int Length);

// Extraction des slots : scan horizontal puis vertical, runs de cases blanches >= 2.
static List<Slot> ExtractSlots(CrosswordGrid g)
{
    var slots = new List<Slot>();
    // Horizontal : pour chaque ligne, runs de cases blanches.
    for (int r = 0; r < g.Rows; r++)
    {
        int c = 0;
        while (c < g.Cols)
        {
            if (g.IsWhite(r, c))
            {
                int start = c;
                while (c < g.Cols && g.IsWhite(r, c)) c++;
                if (c - start >= 2) slots.Add(new Slot(r, start, 'H', c - start));
            }
            else c++;
        }
    }
    // Vertical : pour chaque colonne, runs de cases blanches.
    for (int c = 0; c < g.Cols; c++)
    {
        int r = 0;
        while (r < g.Rows)
        {
            if (g.IsWhite(r, c))
            {
                int start = r;
                while (r < g.Rows && g.IsWhite(r, c)) r++;
                if (r - start >= 2) slots.Add(new Slot(start, c, 'V', r - start));
            }
            else r++;
        }
    }
    return slots;
}

// Grille exemple 5x5 (X = noire, . = blanche).
bool[,] w = {
    { true,  true,  true,  true,  false },
    { true,  false, true,  false, true  },
    { true,  true,  true,  true,  true  },
    { true,  false, true,  false, true  },
    { false, true,  true,  true,  true  },
};
var grid = new CrosswordGrid(w);
var slots = ExtractSlots(grid);
$"Grille {grid.Rows}x{grid.Cols} : {slots.Count} slots ({slots.Count(s => s.Dir=='H')} H, {slots.Count(s => s.Dir=='V')} V)".Display();
var sb = new StringBuilder();
sb.AppendLine("Slots extraits :");
foreach (var s in slots)
    sb.AppendLine($"  ({s.Row},{s.Col}) {s.Dir} longueur {s.Length}");
Show(sb.ToString());


Grille 5x5 : 6 slots (3 H, 3 V)

Slots extraits :
  (0,0) H longueur 4
  (2,0) H longueur 5
  (4,1) H longueur 4
  (0,0) V longueur 4
  (0,2) V longueur 5
  (1,4) V longueur 4


### 1.1 Intersections entre slots

Deux slots se croisent ssi l'un est horizontal, l'autre vertical, et la case de croisement appartient aux deux. A l'intersection, la lettre du slot H a une position $i$ et celle du slot V une position $j$ ; elles doivent etre egales dans toute solution.

In [2]:
// Intersection de deux slots : (position dans slot1, position dans slot2) ou null.
static (int i1, int i2)? Intersect(Slot a, Slot b)
{
    // Un horizontal, l'autre vertical.
    Slot h = a.Dir == 'H' ? a : b;
    Slot v = a.Dir == 'H' ? b : a;
    if (a.Dir == b.Dir) return null;
    // Le slot H couvre la ligne h.Row, colonnes h.Col .. h.Col+h.Length-1.
    // Le slot V couvre la colonne v.Col, lignes v.Row .. v.Row+v.Length-1.
    // Croisement a la case (h.Row, v.Col) si v.Col dans [h.Col, h.Col+h.Length-1]
    // et h.Row dans [v.Row, v.Row+v.Length-1].
    if (v.Col >= h.Col && v.Col < h.Col + h.Length &&
        h.Row >= v.Row && h.Row < v.Row + v.Length)
    {
        int iH = v.Col - h.Col;   // position dans le slot H
        int iV = h.Row - v.Row;   // position dans le slot V
        // Retourner dans l'ordre (a, b).
        return a.Dir == 'H' ? (iH, iV) : (iV, iH);
    }
    return null;
}

// Pre-calcul de toutes les intersections d'une liste de slots.
// intersections[(i,j)] = (pos dans slot i, pos dans slot j).
static Dictionary<(int,int),(int,int)> AllIntersections(List<Slot> slots)
{
    var inter = new Dictionary<(int,int),(int,int)>();
    for (int i = 0; i < slots.Count; i++)
        for (int j = i+1; j < slots.Count; j++)
        {
            var x = Intersect(slots[i], slots[j]);
            if (x is (int p1, int p2)) inter[(i,j)] = (p1,p2);
        }
    return inter;
}

var inter = AllIntersections(slots);
$"Intersections detectees : {inter.Count} (paires de slots se croisant)".Display();
foreach (var kv in inter.Take(5))
    $"  slots[{kv.Key.Item1}] x slots[{kv.Key.Item2}] : pos {kv.Value.Item1} = pos {kv.Value.Item2}".Display();


Intersections detectees : 7 (paires de slots se croisant)

  slots[0] x slots[3] : pos 0 = pos 0

  slots[0] x slots[4] : pos 2 = pos 0

  slots[1] x slots[3] : pos 0 = pos 2

  slots[1] x slots[4] : pos 2 = pos 2

  slots[1] x slots[5] : pos 4 = pos 1

## 2. Solveur Backtracking (from-scratch)

On assigne les slots un par un. Pour chaque slot, on essaie chaque mot du dictionnaire de la bonne longueur ; on accepte le mot s'il est **compatible** avec les slots deja assignes qui le croisent (meme lettre a l'intersection). On recurse jusqu'a ce que tous les slots soient assignes (succes) ou qu'aucun mot ne convienne (echec -> retour arriere).

**Heuristique MRV (Minimum Remaining Values)** : on choisit a chaque etape le slot non-assigne avec le **moins** de candidats restants, ce qui reduit drastiquement la taille de l'arbre de recherche.

In [3]:
#nullable enable
// Compatible : un mot candidat pour slot 'i' respecte-t-il les intersections avec les slots deja assignes ?
static bool Compatible(int i, string word, List<Slot> slots, Dictionary<(int,int),(int,int)> inter, Dictionary<int,string> assignment)
{
    foreach (var kv in inter)
    {
        int a = kv.Key.Item1, b = kv.Key.Item2;
        int other = -1; int posI = -1, posOther = -1;
        if (a == i) { other = b; posI = kv.Value.Item1; posOther = kv.Value.Item2; }
        else if (b == i) { other = a; posI = kv.Value.Item2; posOther = kv.Value.Item1; }
        else continue;
        if (!assignment.ContainsKey(other)) continue;   // l'autre pas encore assigne
        if (word[posI] != assignment[other][posOther]) return false;
    }
    return true;
}

// Backtracking avec heuristique MRV. Retourne l'assignation complete ou null.
static Dictionary<int,string>? SolveBacktrack(List<Slot> slots, Dictionary<int,List<string>> dictByLen, Dictionary<(int,int),(int,int)> inter)
{
    var assignment = new Dictionary<int,string>();
    bool Recurse()
    {
        if (assignment.Count == slots.Count) return true;   // tous assignes
        // MRV : slot non-assigne avec le moins de candidats compatibles.
        int best = -1; int bestCount = int.MaxValue; List<string>? bestCands = null;
        for (int i = 0; i < slots.Count; i++)
        {
            if (assignment.ContainsKey(i)) continue;
            int len = slots[i].Length;
            var cands = dictByLen.GetValueOrDefault(len, new List<string>())
                        .Where(w => Compatible(i, w, slots, inter, assignment)).ToList();
            if (cands.Count < bestCount) { bestCount = cands.Count; best = i; bestCands = cands; if (bestCount == 0) break; }
        }
        if (bestCands == null || bestCands.Count == 0) return false;
        foreach (var w in bestCands)
        {
            assignment[best] = w;
            if (Recurse()) return true;
            assignment.Remove(best);
        }
        return false;
    }
    return Recurse() ? assignment : null;
}
"Solver pret (backtracking + MRV + verification intersections).".Display();


Solver pret (backtracking + MRV + verification intersections).

### 2.1 Resolution sur la grille exemple

On construit un dictionnaire reduit indexe par longueur, puis on lance le backtracking.

In [4]:
// Dictionnaire reduit indexe par longueur (mots majuscules sans accents).
var dictionary = new Dictionary<int,List<string>> {
    [2] = new(){ "AI", "OK", "GO", "NO", "OR", "AN", "AT", "IN", "IS", "IT", "ON", "UP", "US" },
    [3] = new(){ "CAT", "DOG", "RUN", "SUN", "TOP", "KEY", "ICE", "OLD", "NEW", "BAR", "FOR", "OUT", "TWO", "ONE", "SIX", "TEN", "AGE", "AIR", "ART", "BAD", "BAG", "BED", "BIG", "BIT", "BOX", "BOY", "BUS", "BUY", "CAR", "CUP", "CUT", "DAY", "DRY", "EAT", "END", "EYE", "FAR", "FEW", "FLY", "FUN" },
    [4] = new(){ "CODE", "DATA", "FILE", "GAME", "GRID", "HASH", "HELP", "IDEA", "INFO", "JAZZ", "JOIN", "JUMP", "KEYS", "KIND", "LAKE", "LAMP", "LOAD", "LOOP", "NODE", "OPEN", "PATH", "PLAN", "PLAY", "READ", "REST", "ROOT", "RULE", "SAVE", "SEED", "SHIP", "SHOW", "SIGN", "SIZE", "SLOT", "STAR", "STOP", "TASK", "TEST", "TEXT", "TIME", "TREE", "TYPE", "USER", "VIEW", "WAIT", "WALK", "WAVE", "WIND", "WORD", "WORK", "ZERO" },
    [5] = new(){ "ALPHA", "ARRAY", "BASIC", "BRAIN", "BUILD", "CACHE", "CHAIN", "CHAR", "CHECK", "CLASS", "CLEAR", "CLICK", "CODES", "DEBUG", "DEPTH", "ENTRY", "ERROR", "EVENT", "FALSE", "FETCH", "FIELD", "FILES", "FLAGS", "FLOAT", "FRAME", "GRAPH", "GROUP", "HEAP", "INPUT", "LOGIC", "LOOP", "MATCH", "MODEL", "MOTOR", "NODES", "NORTH", "OBJECT", "ORDER", "OUTPUT", "PARSE", "PHASE", "POINT", "PRINT", "QUEUE", "RANGE", "RIGHT", "ROUND", "SCOPE", "SHARE", "SHEET", "SHIFT", "SIGHT", "SLEEP", "SLOT", "STACK", "STATE", "STORE", "STYLE", "SWING", "TABLE", "TERMS", "THREE", "THROW", "TOPIC", "TRACE", "TRACK", "TRAIN", "TREE", "TRUST", "TRUTH", "TUPLE", "TYPE", "UNCLE", "UNDER", "UNION", "UNTIL", "USAGE", "VALID", "VALUE", "VIDEO", "WORLD" },
};

var solution = SolveBacktrack(slots, dictionary, inter);
if (solution != null)
{
    $"Solution trouvee : {solution.Count} slots assignes".Display();
    foreach (var kv in solution.OrderBy(x => x.Key))
        $"  slot {kv.Key} ({slots[kv.Key].Dir} @ {slots[kv.Key].Row},{slots[kv.Key].Col}, len {slots[kv.Key].Length}) = {kv.Value}".Display();
}
else
{
    "Aucune solution (dictionnaire trop petit ou grille trop contrainte).".Display();
}


Solution trouvee : 6 slots assignes

  slot 0 (H @ 0,0, len 4) = CODE

  slot 1 (H @ 2,0, len 5) = DEPTH

  slot 2 (H @ 4,1, len 4) = SHIP

  slot 3 (V @ 0,0, len 4) = CODE

  slot 4 (V @ 0,2, len 5) = DEPTH

  slot 5 (V @ 1,4, len 4) = SHIP

### 2.2 Affichage de la grille remplie

On reconstruit la grille de lettres depuis l'assignation des slots et on l'affiche en ASCII.

In [5]:
#nullable enable
// Affiche la grille remplie : '#' = case noire, '.' = case blanche isolee, sinon la lettre.
static string RenderFilled(CrosswordGrid g, List<Slot> slots, Dictionary<int,string>? assignment)
{
    char[,] letters = new char[g.Rows, g.Cols];
    for (int r = 0; r < g.Rows; r++)
        for (int c = 0; c < g.Cols; c++) letters[r,c] = g.IsWhite(r,c) ? '.' : '#';
    if (assignment != null)
    {
        foreach (var kv in assignment)
        {
            var s = slots[kv.Key]; var w = kv.Value;
            for (int k = 0; k < s.Length; k++)
            {
                int r = s.Dir == 'H' ? s.Row : s.Row + k;
                int c = s.Dir == 'H' ? s.Col + k : s.Col;
                letters[r,c] = w[k];
            }
        }
    }
    var sb = new StringBuilder();
    for (int r = 0; r < g.Rows; r++)
    {
        for (int c = 0; c < g.Cols; c++) sb.Append(letters[r,c]);
        sb.AppendLine();
    }
    return sb.ToString();
}

Show("Grille remplie :");
Show(RenderFilled(grid, slots, solution));


Grille remplie :

CODE#
O#E#S
DEPTH
E#T#I
#SHIP


## 3. Propagation de contraintes (forward-checking)

Le backtracking pur verifie la compatibilite seulement avec les slots **deja assignes**. Le **forward-checking** va plus loin : apres chaque assignation, on filtre les candidats des slots **non encore assignes** qui croisent le slot courant, et on detecte les **domaines vides** (un slot sans candidat restant) pour couper la recherche plus tot.

On instrumente le solveur pour compter les **noeuds explores** et comparer les deux strategies.

In [6]:
#nullable enable
// Backtracking + forward-checking : apres chaque assignation, on elimine des candidats
// des slots non-assignes croisant le slot courant. Coupe si un domaine devient vide.
static (Dictionary<int,string>? sol, int nodes) SolveForwardCheck(List<Slot> slots, Dictionary<int,List<string>> dictByLen, Dictionary<(int,int),(int,int)> inter)
{
    int nodes = 0;
    var domains = new Dictionary<int, HashSet<string>>();
    for (int i = 0; i < slots.Count; i++)
        domains[i] = new HashSet<string>(dictByLen.GetValueOrDefault(slots[i].Length, new List<string>()));
    var assignment = new Dictionary<int,string>();
    bool Recurse()
    {
        nodes++;
        if (assignment.Count == slots.Count) return true;
        // MRV sur les domaines courants.
        int best = -1; int bestCount = int.MaxValue;
        for (int i = 0; i < slots.Count; i++)
        {
            if (assignment.ContainsKey(i)) continue;
            int cnt = domains[i].Count(d => Compatible(i, d, slots, inter, assignment));
            if (cnt < bestCount) { bestCount = cnt; best = i; if (cnt == 0) break; }
        }
        if (bestCount == 0) return false;
        var cands = domains[best].Where(d => Compatible(best, d, slots, inter, assignment)).ToList();
        foreach (var w in cands)
        {
            assignment[best] = w;
            // Forward-check : sauver et filtrer les domaines des slots croisant 'best'.
            var saved = new Dictionary<int, HashSet<string>>();
            foreach (var kv in inter)
            {
                int a = kv.Key.Item1, b = kv.Key.Item2; int other = -1; int posBest = -1, posOther = -1;
                if (a == best) { other = b; posBest = kv.Value.Item1; posOther = kv.Value.Item2; }
                else if (b == best) { other = a; posBest = kv.Value.Item2; posOther = kv.Value.Item1; }
                else continue;
                if (assignment.ContainsKey(other) || saved.ContainsKey(other)) continue;
                saved[other] = new HashSet<string>(domains[other]);
                domains[other].RemoveWhere(d => d[posOther] != w[posBest]);
                if (domains[other].Count == 0) { assignment.Remove(best); goto restore; }
            }
            if (Recurse()) return true;
            restore:
            foreach (var kv in saved) domains[kv.Key] = kv.Value;
            assignment.Remove(best);
        }
        return false;
    }
    return (Recurse() ? assignment : null, nodes);
}

var (solFC, nodesFC) = SolveForwardCheck(slots, dictionary, inter);
$"Forward-checking : {(solFC != null ? "solution trouvee" : "echec")} en {nodesFC} noeuds".Display();
// Backtracking simple instrumente (meme MRV, sans FC) pour comparaison.
int nodesBT = 0; // on relance un comptage leger via SolveBacktrack instrumente ci-dessous.


Forward-checking : solution trouvee en 7 noeuds

### 3.1 Comparaison backtracking vs forward-checking

Sur des grilles plus grandes ou des dictionnaires plus pauvres, le forward-checking explore beaucoup moins de noeuds : il detecte tôt les culs-de-sac.

In [7]:
#nullable enable
// Backtracking simple re-instrumente pour compter les noeuds (sans forward-checking).
static (Dictionary<int,string>? sol, int nodes) SolveBacktrackCounted(List<Slot> slots, Dictionary<int,List<string>> dictByLen, Dictionary<(int,int),(int,int)> inter)
{
    int nodes = 0;
    var assignment = new Dictionary<int,string>();
    bool Recurse()
    {
        nodes++;
        if (assignment.Count == slots.Count) return true;
        int best = -1, bestCount = int.MaxValue; List<string>? bestCands = null;
        for (int i = 0; i < slots.Count; i++)
        {
            if (assignment.ContainsKey(i)) continue;
            var cands = dictByLen.GetValueOrDefault(slots[i].Length, new List<string>()).Where(w => Compatible(i,w,slots,inter,assignment)).ToList();
            if (cands.Count < bestCount) { bestCount = cands.Count; best = i; bestCands = cands; if (bestCount == 0) break; }
        }
        if (bestCands == null || bestCands.Count == 0) return false;
        foreach (var w in bestCands) { assignment[best] = w; if (Recurse()) return true; assignment.Remove(best); }
        return false;
    }
    return (Recurse() ? assignment : null, nodes);
}

var (solBT, nodesBT2) = SolveBacktrackCounted(slots, dictionary, inter);
var sb2 = new StringBuilder();
sb2.AppendLine(" Strategie          | Noeuds explores | Solution");
sb2.AppendLine(new string('-', 45));
sb2.AppendLine($" Backtracking (MRV) | {nodesBT2,15} | {(solBT != null ? "oui" : "non")}");
sb2.AppendLine($" Forward-checking   | {nodesFC,15} | {(solFC != null ? "oui" : "non")}");
Show(sb2.ToString());
$"Verdict : forward-checking explore {nodesFC} noeuds vs {nodesBT2} pour le backtracking simple (MRV deja actif dans les deux).".Display();


 Strategie          | Noeuds explores | Solution
---------------------------------------------
 Backtracking (MRV) |               8 | oui
 Forward-checking   |               7 | oui


Verdict : forward-checking explore 7 noeuds vs 8 pour le backtracking simple (MRV deja actif dans les deux).

## 4. Generation de grille aleatoire

On genere une grille en placant des cases noires avec une probabilite donnee, puis on verifie qu'elle admet des slots (sinon on regenere). La **densite** de cases noires controle la difficulte.

In [8]:
// Generation aleatoire d'une grille avec densite de cases noires.
static CrosswordGrid GenerateRandomGrid(int rows, int cols, double blackDensity)
{
    var rng = new Random(42);
    bool[,] w;
    do
    {
        w = new bool[rows, cols];
        for (int r = 0; r < rows; r++)
            for (int c = 0; c < cols; c++)
                w[r,c] = rng.NextDouble() >= blackDensity;
    } while (ExtractSlots(new CrosswordGrid(w)).Count == 0);   // au moins 1 slot
    return new CrosswordGrid(w);
}

var g2 = GenerateRandomGrid(6, 6, 0.20);
var slots2 = ExtractSlots(g2);
$"Grille aleatoire 6x6 (densite noire ~0.20) : {slots2.Count} slots".Display();
Show("Grille vide (# = noire, . = blanche) :");
Show(RenderFilled(g2, slots2, null));


Grille aleatoire 6x6 (densite noire ~0.20) : 14 slots

Grille vide (# = noire, . = blanche) :

.##.#.
..#...
.....#
...##.
..#.#.
..##..


## 5. Exercices

> **Convention C.1** : les stubs s'executent sans erreur (jamais `throw`). Remplir le corps, re-executer, verifier.

### Exercice 1 — Compter les mots elimines par propagation

Etant donne une grille et un dictionnaire, pour chaque slot, compter combien de mots du dictionnaire (de la bonne longueur) sont **elimines** par les intersections avec les slots deja assignes.

**Indice** : pour chaque slot non-assigne, filtrer les mots compatibles avec l'assignation courante et soustraire du total.

In [9]:
// Exercice 1 : nombre total de mots elimines par propagation apres une assignation partielle.
// TODO etudiant : retourner le compte total sur tous les slots non-assignes.
static int CountEliminatedWords(CrosswordGrid g, Dictionary<int,List<string>> dictByLen, List<Slot> slots, Dictionary<(int,int),(int,int)> inter, Dictionary<int,string> assignment)
{
    int total = 0;
    // Indice : pour chaque slot non-assigne, total += (taille domaine initial - candidats compatibles restants).
    return total;   // TODO etudiant
}

"Exercice a completer".Display();


Exercice a completer

### Exercice 2 — Generation de grille optimisee (maximiser les intersections)

Generer une grille qui maximise le nombre d'intersections entre slots (grille plus riche, mots croises plus serres).

**Indice** : balayer plusieurs grilles aleatoires et garder celle avec le plus d'intersections via `AllIntersections`.

In [10]:
// Exercice 2 : grille maximisant le nombre d'intersections parmi N tirages.
// TODO etudiant : retourner la meilleure grille et son compte d'intersections.
static (CrosswordGrid best, int interCount) GenerateOptimizedGrid(int rows, int cols, int trials = 20)
{
    // Indice : GenerateRandomGrid + ExtractSlots + AllIntersections, garder le max.
    return (GenerateRandomGrid(rows, cols, 0.20), 0);   // TODO etudiant
}

"Exercice a completer".Display();


Exercice a completer

### Exercice 3 — Contraintes de theme (reflexion)

Etendre le solveur pour n'utiliser que des mots d'un **theme** donne (ex : informatique). Quelle structure de dictionnaire permet de filtrer efficacement par theme sans parcourir toute la liste ?

**Indice** : indexer le dictionnaire par `(theme, longueur)` plutot que par `longueur` seule.

In [11]:
// Exercice 3 (reflexion) : structure de dictionnaire par theme.
// TODO etudiant : proposer une signature et un stub de filtrage par theme.
// Indice : Dictionary<(string theme, int length), List<string>>
static List<string> WordsByTheme(Dictionary<(string,int),List<string>> themedDict, string theme, int length)
{
    // TODO etudiant : retourner themedDict[(theme, length)] si present, sinon liste vide.
    return new List<string>();   // TODO etudiant
}

"Exercice a completer".Display();


Exercice a completer

## Conclusion

### Ce que vous avez appris

- **Modelisation CSP** — grille, slots (runs maximaux de cases blanches), intersections (cases de croisement partageant une lettre).
- **Extraction des slots** — scan horizontal/vertical, runs de longueur >= 2.
- **Backtracking** — assignation slot par slot, verification de compatibilite aux intersections, heuristique **MRV** (slot le plus contraint d'abord).
- **Forward-checking** — apres chaque assignation, filtrage des domaines des slots croisant le courant ; coupe des culs-de-sac (domaine vide) plus tot que le backtracking pur.
- **Generation aleatoire** — densite de cases noires controle la difficulte.

### Pont avec la version Python

La version Python ([App-16-Crossword-CSP.ipynb](App-16-Crossword-CSP.ipynb)) combine un solveur OR-Tools CP-SAT (boite noire) et des solveurs backtracking + propagation from-scratch. Ce twin C# deroule les solveurs from-scratch en C# pur (BCL .NET 9, 0 NuGet) — les internes que masque OR-Tools — et la viz matplotlib devient un rendu ASCII de la grille. Le notebook [App-15-SportsScheduling](App-15-SportsScheduling.ipynb) couvre un autre CSP (planification sportive).

### Parite #4956

Twin de parite legitime (Prong B) : OR-Tools CP-SAT cote Python vs backtracking + forward-checking from-scratch cote C#. L'interet est la visibilite des internes (extraction de slots, MRV, forward-checking) et la confirmation que les deux approches convergent vers la meme solution sur la grille exemple.

---
*Marathon #4956 (parite .NET <-> Python).*
